This notebook focuses on parsing macOS system logs into structured events. The goal is not analysis or detection, but reliable extraction of consistent fields from unstructured, multi-line log entries.

Jan 23 00:30:21 Peters-Air syslogd[388]: ASL Sender Statistics
Jan 23 00:30:22 Peters-Air syslogd[388]: Configuration Notice:
	ASL Module "com.apple.cdscheduler" claims selected messages.
	Those messages may not appear in standard system log files or in the ASL database.
Jan 23 00:30:22 Peters-Air syslogd[388]: Configuration Notice:
	ASL Module "com.apple.install" claims selected messages.
	Those messages may not appear in standard system log files or in the ASL database.

Above we can see the raw log and example of logs notices/alerts that have come through. So, we need to break it down. Timestamp, device, service, process ID (PID), message.
We need to consider things like; when does a new log start, what is the delimiter for the message, how can we determine the split between each section of the log etc.

An important first step in breaking down the logs is to extract each log. Where does it start and end? This is easy as we know each log starts with a timestamp. Therefore, multi-line log messages don't matter, these are all included as the end of the message is when the next log starts i.e. where the next timestamp is.

Let's get into the code:

In [2]:
from pathlib import Path
import re
from datetime import datetime
import pandas as pd

In [3]:
log_path = Path("../datasets/raw/system.log")

with log_path.open("r") as f:
    lines = f.readlines()

len(lines)

84

In [4]:
for line in lines[:10]:
    print(line.rstrip())

Jan 23 00:14:01 Peters-Air syslogd[388]: ASL Sender Statistics
Jan 23 00:30:21 Peters-Air syslogd[388]: ASL Sender Statistics
Jan 23 00:30:22 Peters-Air syslogd[388]: Configuration Notice:
	ASL Module "com.apple.cdscheduler" claims selected messages.
	Those messages may not appear in standard system log files or in the ASL database.
Jan 23 00:30:22 Peters-Air syslogd[388]: Configuration Notice:
	ASL Module "com.apple.install" claims selected messages.
	Those messages may not appear in standard system log files or in the ASL database.
Jan 23 00:30:22 Peters-Air syslogd[388]: Configuration Notice:
	ASL Module "com.apple.authd" sharing output destination "/var/log/asl" with ASL Module "com.apple.asl".


In [5]:
header_pattern = re.compile(
    r"^(?P<month>\w{3})\s+"         
    r"(?P<day>\d{1,2})\s+"         
    r"(?P<time>\d{2}:\d{2}:\d{2})\s+"  
    r"(?P<host>\S+)\s+"            
    r"(?P<process>[^\[]+)"          
    r"\[(?P<pid>\d+)\]:\s*"         
)

In [6]:
events = []
current_event = None
current_message = []

for line in lines:
    match = header_pattern.match(line)

    if match:
        if current_event:
            current_event["message"] = " ".join(current_message).strip()
            events.append(current_event)

        current_event = match.groupdict()
        message_part = line[match.end():].strip()
        current_message = [message_part] if message_part else []

    else:
        if current_event:
            current_message.append(line.strip())

if current_event:
    current_event["message"] = " ".join(current_message).strip()
    events.append(current_event)   

In [7]:
len(events)
events[0]

{'month': 'Jan',
 'day': '23',
 'time': '00:14:01',
 'host': 'Peters-Air',
 'process': 'syslogd',
 'pid': '388',
 'message': 'ASL Sender Statistics'}

In [8]:
df = pd.DataFrame(events)
df.head()

,month,day,time,host,process,pid,message
0,Jan,23,00:14:01,Peters-Air,syslogd,388,ASL Sender Statistics
1,Jan,23,00:30:21,Peters-Air,syslogd,388,ASL Sender Statistics
2,Jan,23,00:30:22,Peters-Air,syslogd,388,"Configuration Notice: ASL Module ""com.apple.cd..."
3,Jan,23,00:30:22,Peters-Air,syslogd,388,"Configuration Notice: ASL Module ""com.apple.in..."
4,Jan,23,00:30:22,Peters-Air,syslogd,388,"Configuration Notice: ASL Module ""com.apple.au..."


In [9]:
CURRENT_YEAR = datetime.now().year 

def parse_timestamp(row):
    ts = f"{row['month']} {row['day']} {row ['time']} {CURRENT_YEAR}"
    return datetime.strptime(ts, "%b %d %H:%M:%S %Y")

df["timestamp"] = df.apply(parse_timestamp, axis=1)


In [10]:
df = df.drop(columns= ["month", "day", "time"])
df = df.rename(columns={"process": "process_name"})

df.head()

,host,process_name,pid,message,timestamp
0,Peters-Air,syslogd,388,ASL Sender Statistics,2026-01-23 00:14:01
1,Peters-Air,syslogd,388,ASL Sender Statistics,2026-01-23 00:30:21
2,Peters-Air,syslogd,388,"Configuration Notice: ASL Module ""com.apple.cd...",2026-01-23 00:30:22
3,Peters-Air,syslogd,388,"Configuration Notice: ASL Module ""com.apple.in...",2026-01-23 00:30:22
4,Peters-Air,syslogd,388,"Configuration Notice: ASL Module ""com.apple.au...",2026-01-23 00:30:22


In [13]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["timestamp"] = df["timestamp"].dt.tz_localize("Europe/London").dt.tz_convert("UTC")

In [14]:
df.head()

,host,process_name,pid,message,timestamp
0,Peters-Air,syslogd,388,ASL Sender Statistics,2026-01-23 00:14:01+00:00
1,Peters-Air,syslogd,388,ASL Sender Statistics,2026-01-23 00:30:21+00:00
2,Peters-Air,syslogd,388,"Configuration Notice: ASL Module ""com.apple.cd...",2026-01-23 00:30:22+00:00
3,Peters-Air,syslogd,388,"Configuration Notice: ASL Module ""com.apple.in...",2026-01-23 00:30:22+00:00
4,Peters-Air,syslogd,388,"Configuration Notice: ASL Module ""com.apple.au...",2026-01-23 00:30:22+00:00


In [15]:
output_path = Path("../datasets/processed/system_events.csv")
df.to_csv(output_path, index=False)